# Message Trainer

> Fill in a module description here

In [ ]:
#| default_exp trainers.msg_trainer

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import torch
import torch.nn as nn

## lightning trainer

In [ ]:
# #| export
# from typing import List, Callable, Union, Any, TypeVar, Tuple

# Tensor = TypeVar('torch.tensor')

In [ ]:
# #| export
# import os
# import math
# import torch
# from torch import optim
# from c3jepa_wm.models.msg_net import BaseVAE
# from c3jepa_wm.data.utils import data_loader
# import pytorch_lightning as pl
# from torchvision import transforms
# import torchvision.utils as vutils
# from torchvision.datasets import CelebA
# from torch.utils.data import DataLoader


# class VAEXperiment(pl.LightningModule):

#     def __init__(self,
#                  vae_model: BaseVAE,
#                  params: dict) -> None:
#         super(VAEXperiment, self).__init__()

#         self.model = vae_model
#         self.params = params
#         self.curr_device = None
#         self.hold_graph = False
#         try:
#             self.hold_graph = self.params['retain_first_backpass']
#         except:
#             pass

#     def forward(self, input: Tensor, **kwargs) -> Tensor:
#         return self.model(input, **kwargs)

#     def training_step(self, batch, batch_idx, optimizer_idx = 0):
#         real_img, labels = batch
#         self.curr_device = real_img.device

#         results = self.forward(real_img, labels = labels)
#         train_loss = self.model.loss_function(*results,
#                                               M_N = self.params['kld_weight'], #al_img.shape[0]/ self.num_train_imgs,
#                                               optimizer_idx=optimizer_idx,
#                                               batch_idx = batch_idx)

#         self.log_dict({key: val.item() for key, val in train_loss.items()}, sync_dist=True)

#         return train_loss['loss']

#     def validation_step(self, batch, batch_idx, optimizer_idx = 0):
#         real_img, labels = batch
#         self.curr_device = real_img.device

#         results = self.forward(real_img, labels = labels)
#         val_loss = self.model.loss_function(*results,
#                                             M_N = 1.0, #real_img.shape[0]/ self.num_val_imgs,
#                                             optimizer_idx = optimizer_idx,
#                                             batch_idx = batch_idx)

#         self.log_dict({f"val_{key}": val.item() for key, val in val_loss.items()}, sync_dist=True)

        
#     def on_validation_end(self) -> None:
#         self.sample_images()
        
#     def sample_images(self):
#         # Get sample reconstruction image            
#         test_input, test_label = next(iter(self.trainer.datamodule.test_dataloader()))
#         test_input = test_input.to(self.curr_device)
#         test_label = test_label.to(self.curr_device)

# #         test_input, test_label = batch
#         recons = self.model.generate(test_input, labels = test_label)
#         vutils.save_image(recons.data,
#                           os.path.join(self.logger.log_dir , 
#                                        "Reconstructions", 
#                                        f"recons_{self.logger.name}_Epoch_{self.current_epoch}.png"),
#                           normalize=True,
#                           nrow=12)

#         try:
#             samples = self.model.sample(144,
#                                         self.curr_device,
#                                         labels = test_label)
#             vutils.save_image(samples.cpu().data,
#                               os.path.join(self.logger.log_dir , 
#                                            "Samples",      
#                                            f"{self.logger.name}_Epoch_{self.current_epoch}.png"),
#                               normalize=True,
#                               nrow=12)
#         except Warning:
#             pass

#     def configure_optimizers(self):

#         optims = []
#         scheds = []

#         optimizer = optim.Adam(self.model.parameters(),
#                                lr=self.params['LR'],
#                                weight_decay=self.params['weight_decay'])
#         optims.append(optimizer)
#         # Check if more than 1 optimizer is required (Used for adversarial training)
#         try:
#             if self.params['LR_2'] is not None:
#                 optimizer2 = optim.Adam(getattr(self.model,self.params['submodel']).parameters(),
#                                         lr=self.params['LR_2'])
#                 optims.append(optimizer2)
#         except:
#             pass

#         try:
#             if self.params['scheduler_gamma'] is not None:
#                 scheduler = optim.lr_scheduler.ExponentialLR(optims[0],
#                                                              gamma = self.params['scheduler_gamma'])
#                 scheds.append(scheduler)

#                 # Check if another scheduler is required for the second optimizer
#                 try:
#                     if self.params['scheduler_gamma_2'] is not None:
#                         scheduler2 = optim.lr_scheduler.ExponentialLR(optims[1],
#                                                                       gamma = self.params['scheduler_gamma_2'])
#                         scheds.append(scheduler2)
#                 except:
#                     pass
#                 return optims, scheds
#         except:
#             return optims

In [ ]:
# #| export
# import os
# import yaml
# import argparse
# import numpy as np
# from pathlib import Path
# from c3jepa_wm.models.msg_net import VQVAE
# import torch.backends.cudnn as cudnn
# from pytorch_lightning import Trainer
# from pytorch_lightning.loggers import TensorBoardLogger
# from lightning_fabric.utilities.seed import seed_everything
# from pytorch_lightning.callbacks import LearningRateMonitor, ModelCheckpoint
# from c3jepa_wm.data.vae_datasets import VAEDataset
# from pytorch_lightning.strategies import DDPStrategy

# from lightning.pytorch.loggers import WandbLogger


# parser = argparse.ArgumentParser(description='Generic runner for VAE models')
# parser.add_argument('--config',  '-c',
#                     dest="filename",
#                     metavar='FILE',
#                     help =  'path to the config file',
#                     default='configs/vae.yaml')

# args = parser.parse_args()
# with open(args.filename, 'r') as file:
#     try:
#         config = yaml.safe_load(file)
#     except yaml.YAMLError as exc:
#         print(exc)


# # tb_logger =  TensorBoardLogger(save_dir=config['logging_params']['save_dir'],
# #                                name=config['model_params']['name'],)

# wandb_logger = WandbLogger(save_dir=config['logging_params']['save_dir'],
#                            name=config['model_params']['name'],
#                            log_model="all",)
# wandb_logger.experiment.config["log_dir"] = config['logging_params']['save_dir']

# # For reproducibility
# seed_everything(config['exp_params']['manual_seed'], True)

# model_params = {k: v for k, v in config['model_params'].items() if k != 'name'}
# model = VQVAE(**model_params)

# experiment = VAEXperiment(model,
#                           config['exp_params'])

# data = VAEDataset(**config["data_params"], pin_memory=len(config['trainer_params']['gpus']) != 0)

# data.setup()
# runner = Trainer(logger=wandb_logger,
#                  callbacks=[
#                      LearningRateMonitor(),
#                      ModelCheckpoint(save_top_k=2, 
#                                      dirpath =os.path.join("." , "checkpoints"), 
#                                      monitor= "val_loss",
#                                      save_last= True),
#                  ],
#                  strategy=DDPStrategy(find_unused_parameters=False),
#                  **config['trainer_params'])


# Path(f"{wandb_logger.log_dir}/Samples").mkdir(exist_ok=True, parents=True)
# Path(f"{wandb_logger.log_dir}/Reconstructions").mkdir(exist_ok=True, parents=True)


# print(f"======= Training {config['model_params']['name']} =======")
# runner.fit(experiment, datamodule=data)

## Pure Torch Trainer

In [ ]:
#| export
import os
import torch
import torchvision.utils as vutils
import wandb


class VQVAETrainer:

    def __init__(self, model, params, device, save_dir):
        self.model = model.to(device)
        self.params = params
        self.device = device
        self.save_dir = save_dir

        # Match the optimizer configuration from your original script
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=1e-3)

        # Create output directories for visual inspection
        os.makedirs(os.path.join(self.save_dir, "Reconstructions"), exist_ok=True)

    def train_epoch(self, dataloader, epoch):
        self.model.train()
        total_loss = 0.0

        for batch_idx, batch in enumerate(dataloader):
            # Manually move data to target device (Lightning did this automatically)
            real_img = batch.to(self.device)

            self.optimizer.zero_grad()

            # Forward pass
            recons, input_img, vq_loss = self.model(real_img)

            # Call your model's built-in VQ-VAE loss evaluation function
            loss_dict = self.model.loss_function(
                recons,
                input_img,
                vq_loss,
                M_N=self.params.get("kld_weight", 1.0),
                batch_idx=batch_idx,
            )

            loss = loss_dict["loss"]
            loss.backward()
            self.optimizer.step()

            total_loss += loss.item()

            # Log step-level metrics directly to WandB
            wandb.log({f"train_{k}": v.item() for k, v in loss_dict.items()})

        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch:02d} | Train Loss: {avg_loss:.4f}")
        return avg_loss

    @torch.no_grad()
    def validate_epoch(self, dataloader, epoch):
        self.model.eval()
        total_loss = 0.0

        for batch_idx, batch in enumerate(dataloader):
            real_img = batch.to(self.device)

            recons, input_img, vq_loss = self.model(real_img)
            loss_dict = self.model.loss_function(
                recons, input_img, vq_loss, M_N=1.0, batch_idx=batch_idx
            )

            total_loss += loss_dict["loss"].item()

        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch:02d} | Val Loss:   {avg_loss:.4f}")

        # Log epoch-level validation loss
        wandb.log({"val_loss": avg_loss, "epoch": epoch})
        return avg_loss

    @torch.no_grad()
    def sample_and_save_images(self, test_loader, epoch):
        """Replicates on_validation_end by reconstructing a fixed test batch."""
        self.model.eval()

        # Grab the first batch of images from your test/val loader
        test_input = next(iter(test_loader)).to(self.device)

        # Reconstruct images using VQ-VAE decoder
        recons = self.model.generate(test_input)

        # Save grid image to disk
        recons_path = os.path.join(
            self.save_dir, "Reconstructions", f"recons_Epoch_{epoch}.png"
        )
        vutils.save_image(recons.data, recons_path, normalize=True, nrow=8)

        # Upload the reconstructed image grid directly into WandB panel
        wandb.log(
            {"Visual Reconstructions": wandb.Image(recons_path, caption=f"Epoch {epoch}")}
        )


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()